# Act 1 — Meet the data

> **Opening question:** *"Imagine you just picked up a file of 1,460 house sale records from Ames, Iowa. Before reading a single entry — how many columns describe each house? Are any details missing? And just how different are these houses from each other?"*

---

Before we can answer *what drives house prices*, we need to understand *what we're working with*.

This act answers three questions:
1. **How big is this dataset?** — rows, columns, memory
2. **What kind of information does it hold?** — 80 features describing every aspect of each house
3. **Is anything missing?** — which columns have gaps and how serious are they

In [ ]:
import sys
sys.path.append('..')
from src.utils import *

act_header(
    act_num=1,
    title="Meet the data",
    opening_question="What are we working with — 80 columns about every house in Ames, Iowa. Let's take inventory."
)

## 1.1 — Load the Ames Housing dataset

Download from: https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques/data

Place `train.csv` in `data/raw/` before running this cell.

In [ ]:
df = load_raw('train.csv')
df.head(3)

## 1.2 — Plain-English snapshot
This prints what a non-technical reader needs to know before touching the data.

In [ ]:
data_snapshot(df)

## 1.3 — What types of information do we have?

The Ames dataset has 80 features — that's a lot. They fall into roughly four groups:
- **Size features** — square footage, lot size, number of rooms
- **Quality features** — condition ratings, material grades
- **Location features** — neighbourhood, zoning, proximity to things
- **Time features** — year built, year sold, year remodelled

In [ ]:
plot_dtype_bar(df)

In [ ]:
# The key columns we'll use across all six acts
key_cols = {
    'SalePrice':    'What the house sold for (our target variable)',
    'GrLivArea':    'Above-ground living area in sq ft',
    'OverallQual':  'Overall material and finish quality (1–10 scale)',
    'TotalBsmtSF':  'Total basement area in sq ft',
    'GarageArea':   'Garage size in sq ft',
    'YearBuilt':    'Original construction year',
    'YrSold':       'Year the house was sold',
    'Neighborhood': 'Physical location within Ames city limits',
    'LotArea':      'Lot size in sq ft'
}

print("\n  KEY COLUMNS WE WILL USE:\n")
for col, desc in key_cols.items():
    if col in df.columns:
        print(f"  {col:<16} → {desc}")

## 1.4 — Where is data missing?

Each purple stripe is a missing value. In the Ames dataset many 'missing' values
actually mean 'not applicable' — e.g. PoolQC is missing because most houses have no pool.

In [ ]:
plot_missing_heatmap(df)

In [ ]:
# Show top missing columns with interpretation
missing = df.isnull().mean().sort_values(ascending=False)
missing = missing[missing > 0]

ames_na_meanings = {
    'PoolQC':       'NA = No pool (most houses)',
    'MiscFeature':  'NA = No misc feature',
    'Alley':        'NA = No alley access',
    'Fence':        'NA = No fence',
    'FireplaceQu':  'NA = No fireplace',
    'GarageType':   'NA = No garage',
    'GarageFinish': 'NA = No garage',
    'BsmtQual':     'NA = No basement',
    'BsmtCond':     'NA = No basement',
    'MasVnrType':   'True missing — needs imputation',
    'LotFrontage':  'True missing — linear feet of street'
}

print("\n  MISSING VALUE INTERPRETATION:\n")
for col in missing.index:
    meaning = ames_na_meanings.get(col, 'Review needed')
    print(f"  {col:<16} {missing[col]*100:5.1f}% missing → {meaning}")

In [ ]:
insight_callout(
    "Most 'missing' values in Ames are not errors — they mean the feature doesn't exist.\n"
    "PoolQC is 99% missing because 99% of houses have no pool.\n"
    "The genuinely missing values are LotFrontage and MasVnrType — those need imputation.",
    label="Missing ≠ broken"
)

## 1.5 — Quick statistical profile of key columns

In [ ]:
key_numeric = ['SalePrice', 'GrLivArea', 'TotalBsmtSF', 'GarageArea', 'LotArea', 'OverallQual', 'YearBuilt']
existing_numeric = [c for c in key_numeric if c in df.columns]
df[existing_numeric].describe().round(0)

## 1.6 — Clean and save for Act 2

In [ ]:
df_clean = df.copy()

# Fill 'NA means no feature' columns with 'None'
na_means_none = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
                 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
                 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
                 'MasVnrType']
for col in na_means_none:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna('None')

# Fill numeric NAs with 0 (means no feature)
na_numeric_zero = ['GarageYrBlt', 'GarageArea', 'GarageCars',
                   'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF',
                   'BsmtFullBath', 'BsmtHalfBath', 'MasVnrArea']
for col in na_numeric_zero:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna(0)

# LotFrontage — fill with neighbourhood median (smarter than global median)
if 'LotFrontage' in df_clean.columns and 'Neighborhood' in df_clean.columns:
    df_clean['LotFrontage'] = df_clean.groupby('Neighborhood')['LotFrontage'].transform(
        lambda x: x.fillna(x.median())
    )

# Electrical — one missing, fill with most common
if 'Electrical' in df_clean.columns:
    df_clean['Electrical'] = df_clean['Electrical'].fillna(df_clean['Electrical'].mode()[0])

remaining_missing = df_clean.isnull().sum().sum()
print(f"  Rows before: {len(df):,}")
print(f"  Rows after:  {len(df_clean):,}")
print(f"  Remaining missing values: {remaining_missing}")

save_processed(df_clean, 'cleaned.csv')

---
## Act 1 — Closing punchline

In [ ]:
punchline(
    f"We have {len(df):,} house sales across Ames, Iowa described by 80 features. "
    "Most 'missing' values aren't errors — they mean the feature doesn't exist for that house. "
    "The dataset is largely clean and ready. In Act 2, we find out what 'typical' looks like."
)

**Next → [Act 2 — Distribution Portraits](02_act2_distributions.ipynb)**